# 10 — Build the analytical panel

Merge trade, demand and refinery output into one annual product panel and derive transparent dependence metrics.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.jodi import build_monthly_panel, canonicalise_secondary, filter_portugal_fuels, read_secondary_zip
from portugal_refining_resilience.config import load_analysis_config
from portugal_refining_resilience.metrics import add_supply_metrics, add_yoy
from portugal_refining_resilience.validation import assert_nonnegative, assert_unique


In [ ]:
# The annual panel is built from Eurostat, which runs from 1990 for both
# countries. JODI begins only in 2002, so using it here would cap the annual
# window twelve years short of the available evidence. JODI remains the monthly
# source and is reconciled against this panel over the years both cover.
config = load_analysis_config(ROOT)
start_year, end_year = int(config["start_year"]), int(config["end_year"])

balance = pd.read_csv(PATHS.processed / "eurostat_physical_balance_panel.csv")
panel = balance.loc[
    balance["country"].eq("PT") & balance["year"].between(start_year, end_year)
].copy()
panel = panel[[
    "year", "product", "imports_kt", "exports_kt", "demand_kt", "refinery_output_kt"
]]

# The regime table is keyed by year alone and starts at 2005. Sines and
# Matosinhos both operated throughout the earlier years of the window, so those
# years are labelled two_refineries; Matosinhos closed in 2021 and the later
# labels come from the event table.
regime = pd.read_csv(PATHS.processed / "refining_regime_annual.csv")
panel = panel.merge(regime[["year", "refining_regime"]], on="year", how="left")
panel["refining_regime"] = panel["refining_regime"].fillna("two_refineries")

panel = add_supply_metrics(panel)
panel = add_yoy(panel, ["imports_kt", "exports_kt", "demand_kt", "refinery_output_kt", "net_imports_kt"])
assert_nonnegative(panel, ["imports_kt", "exports_kt", "demand_kt", "refinery_output_kt"])
assert_unique(panel, ["year", "product"])
persist_dataframe(panel, PATHS.processed / "fuel_annual_analytical_panel.csv", key_columns=["year", "product"])
print(f"annual panel: {int(panel['year'].min())}-{int(panel['year'].max())}, "
      f"{len(panel)} rows, {panel['product'].nunique()} products")
display(panel.tail(10))

In [ ]:
raw_zip = PATHS.raw / "jodi" / "world_secondary_csv.zip"
if raw_zip.exists():
    raw = read_secondary_zip(raw_zip)
    canonical = canonicalise_secondary(raw)
    selected = filter_portugal_fuels(canonical)
    monthly_panel = build_monthly_panel(selected)
    persist_dataframe(
        monthly_panel,
        PATHS.processed / "fuel_monthly_analytical_panel.csv",
        key_columns=["date", "product"],
        metadata={"unit": "kt", "event_phases": "May 2021 closure; March 2022 energy stress"},
    )
    display(monthly_panel.tail())
else:
    print("Raw JODI ZIP not found; monthly event-timing panel remains unavailable.")


In [ ]:
missingness = panel.isna().groupby(panel["product"]).mean().T
missingness.to_csv(PATHS.metrics / "annual_panel_missingness.csv")
display(missingness)
